# 固定单步决策：五日预期收益与一次换仓费用诊断

证据来源：Qlib Recorder `9a447c33dc934baeb1d4c3f2b9bf2122`。

只使用已经保存的预测、协方差、预算和决策前持仓。分析四个开发期方案每年首个信号及所选方案2023年起始信号，共58个固定决策点。零费用求解只识别**同一决策点**的费前预期目标最优解；随后按原10/25bp费用重新评价该固定目标。**没有生成新收益路径，没有使用这些决策后的已实现收益，没有优化预测期限。**

若目标权重为w、费用占起始NAV比例为f，则期末预期财富为 `(1-f)*(1+w_risky·mu_H5)`。手续费按真实两只风险资产交易金额的绝对值之和计算。

盈亏平衡期限只是将已经冻结的五日期望收益按`h/5`线性外推，并保持目标和初始费用不变、忽略未来交易及风险变化后的代数描述；它不是预测或建议的持有期。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
from pathlib import Path
import json, hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.model.convex_allocation import DPPConvexAllocationSolver
from src.workflow.stage1_economic_action import fee_affordable_target
from src.backtest.position_accounting import settle_target_session
root=Path('/Users/rziiiii/dev/workspace-worktrees/conditional-variance-20261002')
if not root.exists(): root=Path('/Users/rziiiii/dev/workspace')
s=json.loads((root/'research/conditional_variance_20261002/summary.json').read_text())
assert all(hashlib.sha256((root/p).read_bytes()).hexdigest()==v for p,v in s['source_sha256'].items())
assert s['recorder_id']=='9a447c33dc934baeb1d4c3f2b9bf2122'
qlib.init(provider_uri=str(root/s['provider_uri']),region='us');R.set_uri(s['recorder_uri'])
r=R.get_recorder(experiment_id='607243708770492389',recorder_id=s['recorder_id'])
path=r.load_object('analysis/portfolio_replays.pkl');daily=r.load_object('analysis/daily_solver_audits.pkl');development=r.load_object('policy/development_inputs.pkl');outer=r.load_object('policy/diagnostic_inputs.pkl')
all_rows=[];targets=[];failures=[]
def score(w,h,mu,cost):
 ex=fee_affordable_target(w,pretrade_weights=h,cost_bps=cost)
 settled=settle_target_session(h,open_target_weights=ex,asset_returns=[0.,0.,0.],cost_bps=cost)
 fee=settled['fees'];n=1-fee;g=float(w[:2]@mu)
 return {'target':w.tolist(),'gross_expected_h5_growth':g,'fee':fee,'fee_terminal_wealth_drag':fee*(1+g),'net_expected_h5_growth':n*(1+g)-1,'net_invested_expected_growth_component':n*g,'turnover':settled['turnover']}
def compare(chosen,alt):
 # Flat extrapolation diagnostic only: mu(h)=h/5*mu_H5, fixed weights and one upfront fee.
 gross=chosen['gross_expected_h5_growth']-alt['gross_expected_h5_growth'];fee=alt['fee_terminal_wealth_drag']-chosen['fee_terminal_wealth_drag'];net=chosen['net_expected_h5_growth']-alt['net_expected_h5_growth']
 assert abs(gross+fee-net)<1e-12
 extra_fee=alt['fee']-chosen['fee'];growth_adv=alt['net_invested_expected_growth_component']-chosen['net_invested_expected_growth_component']
 horizon=5*extra_fee/growth_adv if extra_fee>0 and growth_adv>0 else None
 return {'chosen_minus_alternative_gross_h5_bps':gross*1e4,'chosen_fee_advantage_terminal_bps':fee*1e4,'chosen_minus_alternative_net_h5_bps':net*1e4,'alternative_extra_upfront_fee_bps':extra_fee*1e4,'alternative_growth_component_advantage_h5_bps':growth_adv*1e4,'linear_frozen_moment_break_even_sessions':horizon}
for variant in development.keys():
 repeat,name=variant
 if repeat!=1:continue
 inp=development[variant]
 for year in range(2016,2023):
  i=np.flatnonzero(inp['signal_dates'].year==year)[0]
  for cost in [10,25]:targets.append((name,'development',cost,inp,i,(1,name,cost)))
name=s['selected']['variant']+'_annual_refit';inp=outer[(1,'annual_refit')]
i=np.flatnonzero(inp['signal_dates'].year==2023)[0]
for cost in [10,25]:targets.append((name,'reused_diagnostic',cost,inp,i,('outer',1,'annual_refit',cost)))
for name,segment,cost,inp,i,dkey in targets:
 date=inp['signal_dates'][i];row=path[(path.repeat_id==1)&(path.variant==name)&(path.cost_bps==cost)&(path.signal_date==date)].iloc[0];details=daily[dkey][i]
 h=row[['pretrade_soxl_weight','pretrade_soxx_weight','pretrade_cash_weight']].to_numpy(float);w=row[['soxl_weight','soxx_weight','cash_weight']].to_numpy(float);mu=inp['means'][i];sigma=inp['covariance'][i];B=float(inp['budget'][i]);q=min(1,np.sqrt(B/sigma[1,1]));null=np.array([0.,q,1-q])
 chosen=score(w,h,mu,cost);scaled_soxx=score(null,h,mu,cost)
 assert abs(1+chosen['net_expected_h5_growth']-row.objective)<1e-12
 assert abs(1+scaled_soxx['net_expected_h5_growth']-details['nulls']['SCALED_SOXX']['objective'])<1e-12
 zero=DPPConvexAllocationSolver(cost_bps=0,budget_mode='external_positive').solve(h,mu,sigma,risk_budget_variance=B,robust_radius=0.,budget_provenance=inp['budget_provenance'][i])
 assert zero['optimality_verified'] and zero['status']=='solved'
 gross_optimal=score(zero['target_net_weights'],h,mu,cost)
 assert gross_optimal['gross_expected_h5_growth']+1e-8>=max(chosen['gross_expected_h5_growth'],scaled_soxx['gross_expected_h5_growth'])
 all_rows.append({'variant':name,'segment':segment,'signal_date':str(date.date()),'cost_bps':cost,'actual_holdings':h.tolist(),'means_H5':mu.tolist(),'covariance_daily':sigma.tolist(),'B':B,'soxx_gross_mu_over_sigma':float(mu[1]/np.sqrt(sigma[1,1])),'soxl_gross_mu_over_sigma':float(mu[0]/np.sqrt(sigma[0,0])),'chosen':chosen,'scaled_soxx':scaled_soxx,'cost_zero_gross_optimal_evaluated_at_actual_cost':gross_optimal,'chosen_vs_scaled_soxx':compare(chosen,scaled_soxx),'chosen_vs_cost_zero_gross_optimal':compare(chosen,gross_optimal),'cost_zero_certificate':zero['solver_audit']['numerical_certificate'],'saved_solver_certificate':details['solver']['numerical_certificate']})
result={'source_recorder':s['recorder_id'],'scope':'Same saved one-step holdings, frozen mu_H5, daily covariance and B. No new return paths; no realized returns used. Cost-zero solve is gross-objective diagnosis only.','fee_accounting':'True sum of absolute risky traded NAV fractions, cash-funded. Net expected wealth=(1-fee)*(1+w_risky dot mu_H5).','break_even_scope':'Algebraically extrapolate the same mu_H5 linearly with h/5, hold both compared targets and their upfront fees fixed, ignore all future trades and changes in risk/moments. Not a forecast, not an optimized horizon, not evidence of long-run superiority.','source_hashes':s['source_sha256'],'decisions':all_rows,'selected_points':len(all_rows)}
Path('/tmp/conditional-variance-one-step-fee-diagnosis.json').write_text(json.dumps(result,indent=2))
summary=[]
for a in all_rows:
 b=a['chosen_vs_scaled_soxx'];z=a['chosen_vs_cost_zero_gross_optimal'];summary.append({'variant':a['variant'],'signal_date':a['signal_date'],'cost_bps':a['cost_bps'],'held_SOXL':a['actual_holdings'][0],'chosen_SOXL':a['chosen']['target'][0],'chosen_SOXX':a['chosen']['target'][1],'scaled_SOXX':a['scaled_soxx']['target'][1],'zero_SOXL':a['cost_zero_gross_optimal_evaluated_at_actual_cost']['target'][0],'zero_SOXX':a['cost_zero_gross_optimal_evaluated_at_actual_cost']['target'][1],**b,'vs_zero_gross_h5_bps':z['chosen_minus_alternative_gross_h5_bps'],'vs_zero_net_h5_bps':z['chosen_minus_alternative_net_h5_bps'],'vs_zero_break_even_sessions':z['linear_frozen_moment_break_even_sessions']})
df=pd.DataFrame(summary);df.to_csv('/tmp/conditional-variance-one-step-fee-diagnosis.csv',index=False)
print('Audited fixed decisions:',len(df));print(df[df.variant.str.startswith('expanding')].to_string(index=False));print('\nCOUNTS:',json.dumps({'scaled_soxx_gross_better':int(df.chosen_minus_alternative_gross_h5_bps.lt(-1e-5).sum()),'scaled_soxx_net_worse':int(df.chosen_minus_alternative_net_h5_bps.gt(1e-5).sum()),'gross_optimum_gross_better':int(df.vs_zero_gross_h5_bps.lt(-1e-5).sum()),'gross_optimum_net_worse':int(df.vs_zero_net_h5_bps.gt(1e-5).sum())}))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[90440:MainThread](2026-10-02 22:56:05,131) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[90440:MainThread](2026-10-02 22:56:05,132) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[90440:MainThread](2026-10-02 22:56:05,132) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/conditional-variance-20261002/data/qlib/published_futu_affine_20260925_v1')}


Audited fixed decisions: 58
                                variant signal_date  cost_bps  held_SOXL  chosen_SOXL  chosen_SOXX  scaled_SOXX  zero_SOXL  zero_SOXX  chosen_minus_alternative_gross_h5_bps  chosen_fee_advantage_terminal_bps  chosen_minus_alternative_net_h5_bps  alternative_extra_upfront_fee_bps  alternative_growth_component_advantage_h5_bps  linear_frozen_moment_break_even_sessions  vs_zero_gross_h5_bps  vs_zero_net_h5_bps  vs_zero_break_even_sessions
             expanding_variance_control  2016-01-04        10   0.000000     0.341497 1.229062e-07     1.000000   0.017457   0.982543                              -0.859344                           6.593833                             5.734489                           6.576201                                       0.841711                                 39.064467             -1.702989            4.891687                    19.519587
             expanding_variance_control  2016-01-04        25   0.000000     0.341497 1.0983

## 结论

58个固定决策点中，SOXX的费前五日均值/日波动率均高于SOXL；零费用全局最优配置的费前预期增长也全部更高，但计入原费用后，全部低于原选择的净预期目标。

2016首次从现金进入时，10bp下原SOXL配置费前预期增长25.507bp、费用3.414bp，净增长22.085bp；全SOXX分别为26.367bp、9.990bp、16.350bp。求解器为了节省入场费用，正确地选择了较低费前增长的杠杆敞口。

2022年首日，原持仓接近32%SOXL。转换至风险可行的94.03%SOXX，五日费前增长优势仅3.954bp，而额外初始费用为12.508bp（10bp费率）；短期目标因此继续偏向SOXL。两固定目标的线性盈亏平衡期限约16.05/40.99个交易日（10/25bp），仅作机制诊断。

这支持“短规划期与一次换仓费用共同造成进入偏好和持仓延续”的解释，不是数值求解错误。风险预算超过全SOXX风险时，费前最优解也可能是SOXL/SOXX混合，而非纯SOXX。单步预期优势不能证明任何长期已实现优势；没有新策略晋升或部署依据。